# Analysis of Emergency Obstetric Care (EmOC) in Kenya, Kisumu
> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../kano/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate isochrones on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd


import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [2]:
# Set paths to access Kano data
# Define directories
data_inputs = '../scripts/Kisumu/data-inputs/'
data_temp = '../scripts/Kisumu/data-temp/'
model_outputs = '../scripts/Kisumu/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined with the assistance of local experts, based on data obtained from the [datasets of health facilities](https://doi.org/10.6084/m9.figshare.22689667.v2).

In [4]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities_kisumu.geojson')
healthcare_facilities_validated

,field_1,Field_1_1,Admin_1,facility_name,facility_type,owner_type_name,latitude,longitude,Sub_Couinty,LL_Source,...,high_dependancy_unit_beds,isolation_beds,general_theatres,maternity_theatres,minor_theatres,BMoc,CeMoc,Local_Validation,hcf_id,geometry
0,1,Kenya,Kisumu,Macmohan Health Care Limited,Basic Health Centre,Private Practice,-0.08287,34.77501,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,0,POINT (34.77501 -0.08287)
1,2,Kenya,Kisumu,"Impact Research, Training and Healthcare Services",Basic Health Centre,Private Practice,-0.08661,34.76615,Kisumu Central,KMHFR,...,0,2,0,False,False,True,True,Private Comprehensive EmOC,1,POINT (34.76615 -0.08661)
2,3,Kenya,Kisumu,Lafe Medical Centre,Medical Clinic,Private Practice,-0.07987,34.78401,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,2,POINT (34.78401 -0.07987)
3,4,Kenya,Kisumu,Jaramogi Odinga Teaching & Referral Hospital,Specialized & Tertiary Referral hospitals,Ministry of Health,-0.08855,34.77038,Kisumu Central,KMHFR,...,0,0,0,False,False,True,True,Public Comprehensive EmOC,3,POINT (34.77038 -0.08855)
4,5,Kenya,Kisumu,Ahava Medical Centre,Medical Clinic,Private Practice,-0.09979,34.78218,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,4,POINT (34.78218 -0.09979)
5,6,Kenya,Kisumu,Ambercare Medical Care,Medical Center,Private Practice,-0.08458,34.76384,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,5,POINT (34.76384 -0.08458)
6,7,Kenya,Kisumu,Tayyiba Medical Centre,Medical Center,Faith Based Organization,-0.10062,34.75038,Kisumu Central,KMHFR,...,0,5,2,True,False,False,True,Private Comprehensive EmOC,6,POINT (34.75038 -0.10062)
7,8,Kenya,Kisumu,St. Francis Hillside Medicare,Medical Center,Private Practice,-0.10008,34.75251,Kisumu Central,KMHFR,...,0,0,1,False,False,True,False,Private Basic EmOC,7,POINT (34.75251 -0.10008)
8,9,Kenya,Kisumu,Kannika International Ltd Hospital(Kisumu),Secondary care hospitals,Private Practice,-0.09100,34.76022,Kisumu Central,KMHFR,...,0,0,1,True,False,True,True,Private Comprehensive EmOC,8,POINT (34.76022 -0.091)
9,10,Kenya,Kisumu,Gifted Minds Medical Centre,Basic Health Centre,Private Practice,-0.07903,34.77245,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,9,POINT (34.77245 -0.07903)


In [5]:
print(healthcare_facilities_validated['BMoc'].unique())
print(healthcare_facilities_validated['CeMoc'].unique())


[ True False]
[False  True]


In [6]:
conditions = [
    (healthcare_facilities_validated['owner_type_name'] == 'Ministry of Health') &
    (healthcare_facilities_validated['CeMoc'] == True),

    (healthcare_facilities_validated['owner_type_name'] == 'Ministry of Health') &
    (healthcare_facilities_validated['BMoc'] == True),

    (healthcare_facilities_validated['owner_type_name'] != 'Ministry of Health') &
    (healthcare_facilities_validated['CeMoc'] == True),

    (healthcare_facilities_validated['owner_type_name'] != 'Ministry of Health') &
    (healthcare_facilities_validated['BMoc'] == True)
]

choices = [
    'Public Comprehensive EmOC',
    'Public Basic EmOC',
    'Private Comprehensive EmOC',
    'Private Basic EmOC'
]

healthcare_facilities_validated['Local_Validation'] = np.select(
    conditions, choices, default=None
)
healthcare_facilities_validated

,field_1,Field_1_1,Admin_1,facility_name,facility_type,owner_type_name,latitude,longitude,Sub_Couinty,LL_Source,...,high_dependancy_unit_beds,isolation_beds,general_theatres,maternity_theatres,minor_theatres,BMoc,CeMoc,Local_Validation,hcf_id,geometry
0,1,Kenya,Kisumu,Macmohan Health Care Limited,Basic Health Centre,Private Practice,-0.08287,34.77501,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,0,POINT (34.77501 -0.08287)
1,2,Kenya,Kisumu,"Impact Research, Training and Healthcare Services",Basic Health Centre,Private Practice,-0.08661,34.76615,Kisumu Central,KMHFR,...,0,2,0,False,False,True,True,Private Comprehensive EmOC,1,POINT (34.76615 -0.08661)
2,3,Kenya,Kisumu,Lafe Medical Centre,Medical Clinic,Private Practice,-0.07987,34.78401,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,2,POINT (34.78401 -0.07987)
3,4,Kenya,Kisumu,Jaramogi Odinga Teaching & Referral Hospital,Specialized & Tertiary Referral hospitals,Ministry of Health,-0.08855,34.77038,Kisumu Central,KMHFR,...,0,0,0,False,False,True,True,Public Comprehensive EmOC,3,POINT (34.77038 -0.08855)
4,5,Kenya,Kisumu,Ahava Medical Centre,Medical Clinic,Private Practice,-0.09979,34.78218,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,4,POINT (34.78218 -0.09979)
5,6,Kenya,Kisumu,Ambercare Medical Care,Medical Center,Private Practice,-0.08458,34.76384,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,5,POINT (34.76384 -0.08458)
6,7,Kenya,Kisumu,Tayyiba Medical Centre,Medical Center,Faith Based Organization,-0.10062,34.75038,Kisumu Central,KMHFR,...,0,5,2,True,False,False,True,Private Comprehensive EmOC,6,POINT (34.75038 -0.10062)
7,8,Kenya,Kisumu,St. Francis Hillside Medicare,Medical Center,Private Practice,-0.10008,34.75251,Kisumu Central,KMHFR,...,0,0,1,False,False,True,False,Private Basic EmOC,7,POINT (34.75251 -0.10008)
8,9,Kenya,Kisumu,Kannika International Ltd Hospital(Kisumu),Secondary care hospitals,Private Practice,-0.09100,34.76022,Kisumu Central,KMHFR,...,0,0,1,True,False,True,True,Private Comprehensive EmOC,8,POINT (34.76022 -0.091)
9,10,Kenya,Kisumu,Gifted Minds Medical Centre,Basic Health Centre,Private Practice,-0.07903,34.77245,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,9,POINT (34.77245 -0.07903)


In [7]:
healthcare_facilities_validated['hcf_id'] = range(len(healthcare_facilities_validated))
healthcare_facilities_validated

,field_1,Field_1_1,Admin_1,facility_name,facility_type,owner_type_name,latitude,longitude,Sub_Couinty,LL_Source,...,high_dependancy_unit_beds,isolation_beds,general_theatres,maternity_theatres,minor_theatres,BMoc,CeMoc,Local_Validation,hcf_id,geometry
0,1,Kenya,Kisumu,Macmohan Health Care Limited,Basic Health Centre,Private Practice,-0.08287,34.77501,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,0,POINT (34.77501 -0.08287)
1,2,Kenya,Kisumu,"Impact Research, Training and Healthcare Services",Basic Health Centre,Private Practice,-0.08661,34.76615,Kisumu Central,KMHFR,...,0,2,0,False,False,True,True,Private Comprehensive EmOC,1,POINT (34.76615 -0.08661)
2,3,Kenya,Kisumu,Lafe Medical Centre,Medical Clinic,Private Practice,-0.07987,34.78401,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,2,POINT (34.78401 -0.07987)
3,4,Kenya,Kisumu,Jaramogi Odinga Teaching & Referral Hospital,Specialized & Tertiary Referral hospitals,Ministry of Health,-0.08855,34.77038,Kisumu Central,KMHFR,...,0,0,0,False,False,True,True,Public Comprehensive EmOC,3,POINT (34.77038 -0.08855)
4,5,Kenya,Kisumu,Ahava Medical Centre,Medical Clinic,Private Practice,-0.09979,34.78218,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,4,POINT (34.78218 -0.09979)
5,6,Kenya,Kisumu,Ambercare Medical Care,Medical Center,Private Practice,-0.08458,34.76384,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,5,POINT (34.76384 -0.08458)
6,7,Kenya,Kisumu,Tayyiba Medical Centre,Medical Center,Faith Based Organization,-0.10062,34.75038,Kisumu Central,KMHFR,...,0,5,2,True,False,False,True,Private Comprehensive EmOC,6,POINT (34.75038 -0.10062)
7,8,Kenya,Kisumu,St. Francis Hillside Medicare,Medical Center,Private Practice,-0.10008,34.75251,Kisumu Central,KMHFR,...,0,0,1,False,False,True,False,Private Basic EmOC,7,POINT (34.75251 -0.10008)
8,9,Kenya,Kisumu,Kannika International Ltd Hospital(Kisumu),Secondary care hospitals,Private Practice,-0.09100,34.76022,Kisumu Central,KMHFR,...,0,0,1,True,False,True,True,Private Comprehensive EmOC,8,POINT (34.76022 -0.091)
9,10,Kenya,Kisumu,Gifted Minds Medical Centre,Basic Health Centre,Private Practice,-0.07903,34.77245,Kisumu Central,KMHFR,...,0,0,0,False,False,True,False,Private Basic EmOC,9,POINT (34.77245 -0.07903)


In [8]:
healthcare_facilities_validated.to_file(data_inputs + 'healthcare_facilities_kisumu.geojson', driver='GeoJSON')

In [16]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities_kisumu.geojson', driver='GeoJSON')

/opt/miniconda3/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [10]:
study_area = gpd.read_file(data_inputs + 'grid-boundary-kisumu.gpkg')
raster_path = data_inputs + 'ken_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [11]:
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_30776/2284915905.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


In [12]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [13]:
with rasterio.open(data_inputs + 'kisumu_ken_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

### Adding population data at 1km grid to 100m grid

In [17]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

epsg = 'EPSG:32632'

In [18]:
# Preparing grid
grid_file = data_inputs + 'grid-boundary-kisumu.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'geometry','latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,"POLYGON ((3465693.407 -99.313, 3465693.407 0, ...",-0.000404,-0.000809,0.000000,34.733131,34.732632,34.733629
1,1,"POLYGON ((3465816.756 -99.314, 3465816.756 0, ...",-0.000404,-0.000809,0.000000,34.734128,34.733629,34.734627
2,2,"POLYGON ((3465940.105 -99.315, 3465940.105 0, ...",-0.000404,-0.000809,0.000000,34.735126,34.734627,34.735625
3,3,"POLYGON ((3466063.456 -99.315, 3466063.456 0, ...",-0.000404,-0.000809,0.000000,34.736124,34.735625,34.736623
4,4,"POLYGON ((3466186.808 -99.316, 3466186.808 0, ...",-0.000404,-0.000809,0.000000,34.737122,34.736623,34.737621
...,...,...,...,...,...,...,...,...
19795,19795,"POLYGON ((3468774.739 -18873.452, 3468774.767 ...",-0.153262,-0.153666,-0.152857,34.758152,34.757652,34.758651
19796,19796,"POLYGON ((3468898.113 -18873.612, 3468898.142 ...",-0.153262,-0.153666,-0.152857,34.759150,34.758650,34.759649
19797,19797,"POLYGON ((3469021.489 -18873.772, 3469021.517 ...",-0.153262,-0.153666,-0.152857,34.760147,34.759648,34.760647
19798,19798,"POLYGON ((3469144.865 -18873.932, 3469144.894 ...",-0.153262,-0.153666,-0.152857,34.761145,34.760646,34.761644


Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.

In [20]:
# Count buildings per grid cell

# Load Google building footprints
building_file = data_inputs + 'Kisumu_GOB.parquet'
buildings = gpd.read_parquet(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

for df in [grid, buildings]:
    cols_to_remove = [c for c in df.columns if c.startswith('index_') or c.endswith('_left') or c.endswith('_right')]
    if cols_to_remove:
        df.drop(columns=cols_to_remove, inplace=True)

# Join buildings to grid using centroid
grid_buildings = grid.sjoin(
    buildings.set_geometry('centroid').drop(columns='geometry'),
    how='inner',
    predicate='intersects'
)

# Count buildings per grid cell
building_counts = grid_buildings.groupby('grid_id').size().rename('bcount')

# Add building count to grid
grid = grid.merge(building_counts, on='grid_id', how='left')
grid['bcount'] = grid['bcount'].fillna(0)   # assign 0 to empty cells
grid

,grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max,bcount
0,0,"POLYGON ((3465693.407 -99.313, 3465693.407 0, ...",-0.000404,-0.000809,0.000000,34.733131,34.732632,34.733629,7.0
1,1,"POLYGON ((3465816.756 -99.314, 3465816.756 0, ...",-0.000404,-0.000809,0.000000,34.734128,34.733629,34.734627,10.0
2,2,"POLYGON ((3465940.105 -99.315, 3465940.105 0, ...",-0.000404,-0.000809,0.000000,34.735126,34.734627,34.735625,1.0
3,3,"POLYGON ((3466063.456 -99.315, 3466063.456 0, ...",-0.000404,-0.000809,0.000000,34.736124,34.735625,34.736623,5.0
4,4,"POLYGON ((3466186.808 -99.316, 3466186.808 0, ...",-0.000404,-0.000809,0.000000,34.737122,34.736623,34.737621,14.0
...,...,...,...,...,...,...,...,...,...
19795,19795,"POLYGON ((3468774.739 -18873.452, 3468774.767 ...",-0.153262,-0.153666,-0.152857,34.758152,34.757652,34.758651,0.0
19796,19796,"POLYGON ((3468898.113 -18873.612, 3468898.142 ...",-0.153262,-0.153666,-0.152857,34.759150,34.758650,34.759649,0.0
19797,19797,"POLYGON ((3469021.489 -18873.772, 3469021.517 ...",-0.153262,-0.153666,-0.152857,34.760147,34.759648,34.760647,0.0
19798,19798,"POLYGON ((3469144.865 -18873.932, 3469144.894 ...",-0.153262,-0.153666,-0.152857,34.761145,34.760646,34.761644,0.0


The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [21]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Load coarse population raster
pop_file = data_path / 'kisumu_ken_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Convert raster to vector population grid
pop_grid = raster2vector(pop_raster, transform, crs)
pop_grid = pop_grid.to_crs(epsg)
pop_grid['pop_grid_id'] = range(len(pop_grid))
pop_grid.to_csv(data_path / 'pop_grid_id.csv')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

Index(['grid_id', 'geometry', 'latitude', 'lat_min', 'lat_max', 'longitude',
       'lon_min', 'lon_max', 'bcount', 'centroid', 'index_right',
       'pop_grid_pop', 'pop_grid_id'],
      dtype='object')


,grid_id,bcount,pop_grid_id,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,7.0,5,"POLYGON ((3465693.407 -99.313, 3465693.407 0, ...",-0.000404,-0.000809,0.0,34.733131,34.732632,34.733629
1,1,10.0,5,"POLYGON ((3465816.756 -99.314, 3465816.756 0, ...",-0.000404,-0.000809,0.0,34.734128,34.733629,34.734627
2,2,1.0,6,"POLYGON ((3465940.105 -99.315, 3465940.105 0, ...",-0.000404,-0.000809,0.0,34.735126,34.734627,34.735625
3,3,5.0,6,"POLYGON ((3466063.456 -99.315, 3466063.456 0, ...",-0.000404,-0.000809,0.0,34.736124,34.735625,34.736623
4,4,14.0,6,"POLYGON ((3466186.808 -99.316, 3466186.808 0, ...",-0.000404,-0.000809,0.0,34.737122,34.736623,34.737621


In [22]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

,grid_id,bcount,pop_grid_id,geometry_x,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,geometry_y,pop
0,0,7.0,5,"POLYGON ((3465693.407 -99.313, 3465693.407 0, ...",-0.000404,-0.000809,0.0,34.733131,34.732632,34.733629,32.0,0.218750,NaN,"POLYGON ((3464781.139 460.478, 3465811.301 460...",NaN
1,1,10.0,5,"POLYGON ((3465816.756 -99.314, 3465816.756 0, ...",-0.000404,-0.000809,0.0,34.734128,34.733629,34.734627,32.0,0.312500,NaN,"POLYGON ((3464781.139 460.478, 3465811.301 460...",NaN
2,2,1.0,6,"POLYGON ((3465940.105 -99.315, 3465940.105 0, ...",-0.000404,-0.000809,0.0,34.735126,34.734627,34.735625,363.0,0.002755,387.382629,"POLYGON ((3465811.301 460.511, 3466841.536 460...",1.067170
3,3,5.0,6,"POLYGON ((3466063.456 -99.315, 3466063.456 0, ...",-0.000404,-0.000809,0.0,34.736124,34.735625,34.736623,363.0,0.013774,387.382629,"POLYGON ((3465811.301 460.511, 3466841.536 460...",5.335849
4,4,14.0,6,"POLYGON ((3466186.808 -99.316, 3466186.808 0, ...",-0.000404,-0.000809,0.0,34.737122,34.736623,34.737621,363.0,0.038567,387.382629,"POLYGON ((3465811.301 460.511, 3466841.536 460...",14.940377


In [23]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])
grid.head()


,grid_id,bcount,pop_grid_id,geometry_x,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop
0,0,7.0,5,"POLYGON ((3465693.407 -99.313, 3465693.407 0, ...",-0.000404,-0.000809,0.0,34.733131,34.732632,34.733629,32.0,0.218750,NaN,NaN
1,1,10.0,5,"POLYGON ((3465816.756 -99.314, 3465816.756 0, ...",-0.000404,-0.000809,0.0,34.734128,34.733629,34.734627,32.0,0.312500,NaN,NaN
2,2,1.0,6,"POLYGON ((3465940.105 -99.315, 3465940.105 0, ...",-0.000404,-0.000809,0.0,34.735126,34.734627,34.735625,363.0,0.002755,387.382629,1.067170
3,3,5.0,6,"POLYGON ((3466063.456 -99.315, 3466063.456 0, ...",-0.000404,-0.000809,0.0,34.736124,34.735625,34.736623,363.0,0.013774,387.382629,5.335849
4,4,14.0,6,"POLYGON ((3466186.808 -99.316, 3466186.808 0, ...",-0.000404,-0.000809,0.0,34.737122,34.736623,34.737621,363.0,0.038567,387.382629,14.940377


In [ ]:
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-kisumu.gpkg', driver='GPKG')

In [ ]:
# Preparing gird centroids with population attribute for accessibility analysis
grid = gpd.read_file(data_temp + "pop-grid-kisumu.gpkg")
grid_ll = grid.to_crs(epsg=4326)

grid_ll["geometry"] = grid_ll.geometry.centroid

grid_ll["latitude"] = grid_ll["geometry"].y
grid_ll["longitude"] = grid_ll["geometry"].x

grid_centroids = grid_ll[["grid_id", "latitude", "longitude", "geometry", "pop"]].copy()
grid_centroids = grid_centroids.set_geometry("geometry")
grid_centroids.set_crs("EPSG:4326", inplace=True)

grid_centroids = grid_centroids.dropna(subset=["pop"])
grid_centroids.reset_index(drop=True, inplace=True)

grid_centroids.to_file(data_temp + "grid_centroids.gpkg", driver="GPKG")

/var/folders/0r/5__qvz317ljbpq7lc75fxm_w0000gn/T/ipykernel_61248/2455005511.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  grid_ll["geometry"] = grid_ll.geometry.centroid


In [ ]:
grid_centroids

,grid_id,latitude,longitude,geometry,pop
0,0,-0.000404,34.733131,POINT (34.73313 -0.0004),NaN
1,1,-0.000404,34.734128,POINT (34.73413 -0.0004),NaN
2,2,-0.000404,34.735126,POINT (34.73513 -0.0004),1.067170
3,3,-0.000404,34.736124,POINT (34.73612 -0.0004),5.335849
4,4,-0.000404,34.737122,POINT (34.73712 -0.0004),14.940377
...,...,...,...,...,...
19795,19795,-0.153262,34.758152,POINT (34.75815 -0.15326),NaN
19796,19796,-0.153262,34.759150,POINT (34.75915 -0.15326),NaN
19797,19797,-0.153262,34.760147,POINT (34.76015 -0.15326),NaN
19798,19798,-0.153262,34.761145,POINT (34.76115 -0.15326),NaN


## 2. Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [3]:
origin_gdf = gpd.read_file(data_temp + "grid_centroids.gpkg")
origin_name_column = 'grid_id'
destination_gdf = gpd.read_file(data_inputs + 'healthcare_facilities_kisumu.geojson').dropna(subset=['geometry'])
destination_name_column = 'hcf_id'

In [4]:
# Extract coordinates
origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))
locations = origins + destinations

In [5]:
# Indices
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

In [7]:
# Prepare API request
body = {
    'locations': locations,
    'destinations': destinations_index,
    'sources': origins_index,
    'metrics': ['distance', 'duration']
}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

# Make request
response = requests.post(
    'https://api.openrouteservice.org/v2/matrix/driving-car',
    json=body,
    headers=headers
)

In [8]:
# Parse response
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities_validated[(destination_gdf.geometry.x == dest_x) & (destination_gdf.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# Convert the results into a DataFrame
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

In [ ]:
# Save to CSV
merged_df = pd.merge(matrix_df, grid_df[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [3]:
# If not loaded yet, read from the temporary folder
centroids_df = gpd.read_file(data_temp +'pop-grid-kisumu.gpkg')
centroids_df

,grid_id,bcount,pop_grid_id,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop,geometry
0,0,7.0,5,-0.000404,-0.000809,0.000000,34.733131,34.732632,34.733629,32.0,0.218750,NaN,NaN,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73..."
1,1,10.0,5,-0.000404,-0.000809,0.000000,34.734128,34.733629,34.734627,32.0,0.312500,NaN,NaN,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73..."
2,2,1.0,6,-0.000404,-0.000809,0.000000,34.735126,34.734627,34.735625,363.0,0.002755,387.382629,1.067170,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73..."
3,3,5.0,6,-0.000404,-0.000809,0.000000,34.736124,34.735625,34.736623,363.0,0.013774,387.382629,5.335849,"POLYGON ((34.73662 -0.00081, 34.73662 0, 34.73..."
4,4,14.0,6,-0.000404,-0.000809,0.000000,34.737122,34.736623,34.737621,363.0,0.038567,387.382629,14.940377,"POLYGON ((34.73762 -0.00081, 34.73762 0, 34.73..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19795,19795,0.0,332,-0.153262,-0.153666,-0.152857,34.758152,34.757652,34.758651,0.0,NaN,21.023420,NaN,"POLYGON ((34.75865 -0.15367, 34.75865 -0.15286..."
19796,19796,0.0,332,-0.153262,-0.153666,-0.152857,34.759150,34.758650,34.759649,0.0,NaN,21.023420,NaN,"POLYGON ((34.75965 -0.15367, 34.75965 -0.15286..."
19797,19797,0.0,333,-0.153262,-0.153666,-0.152857,34.760147,34.759648,34.760647,0.0,NaN,NaN,NaN,"POLYGON ((34.76065 -0.15367, 34.76065 -0.15286..."
19798,19798,0.0,333,-0.153262,-0.153666,-0.152857,34.761145,34.760646,34.761644,0.0,NaN,NaN,NaN,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286..."


In [4]:
# If not loaded yet, read from the temporary folder
matrix_df = pd.read_csv(data_temp +'OD-matrix-kisumu-access-emoc.csv')
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,0,0,698.76,12.26
1,0,1,685.44,12.15
2,0,2,674.14,12.05
3,0,3,651.94,11.87
4,0,4,639.37,11.76
...,...,...,...,...
554395,27,19795,1896.60,27.21
554396,27,19796,1891.80,27.17
554397,27,19797,1881.36,27.08
554398,27,19798,1870.92,27.00


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the 2SFCA without a travel time estimate.

In [5]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,0,0,698.76,12.26
1,0,1,685.44,12.15
2,0,2,674.14,12.05
3,0,3,651.94,11.87
4,0,4,639.37,11.76
...,...,...,...,...
554395,27,19795,1896.60,27.21
554396,27,19796,1891.80,27.17
554397,27,19797,1881.36,27.08
554398,27,19798,1870.92,27.00


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [7]:
matrix_df.columns

Index(['origin_id', 'destination_id', 'duration_seconds', 'distance_km'], dtype='object')

In [8]:
centroids_df.columns

Index(['grid_id', 'bcount', 'pop_grid_id', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max', 'pop_grid_bcount', 'pop_weight',
       'pop_grid_pop', 'pop', 'geometry'],
      dtype='object')

In [9]:
pop_centroids_hcf = pd.merge(
    matrix_df,
    centroids_df[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min',
                  'lon_max', 'lat_max', 'bcount', 'pop_grid_bcount',
                  'pop_grid_pop', 'pop', 'geometry']],
    left_on='destination_id',
    right_on='grid_id',
    how='left'
)


In [10]:
pop_centroids_hcf

,origin_id,destination_id,duration_seconds,distance_km,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,bcount,pop_grid_bcount,pop_grid_pop,pop,geometry
0,0,0,698.76,12.26,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,7.0,32.0,NaN,NaN,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73..."
1,0,1,685.44,12.15,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,10.0,32.0,NaN,NaN,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73..."
2,0,2,674.14,12.05,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,1.0,363.0,387.382629,1.067170,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73..."
3,0,3,651.94,11.87,3,34.736124,-0.000404,34.735625,-0.000809,34.736623,0.000000,5.0,363.0,387.382629,5.335849,"POLYGON ((34.73662 -0.00081, 34.73662 0, 34.73..."
4,0,4,639.37,11.76,4,34.737122,-0.000404,34.736623,-0.000809,34.737621,0.000000,14.0,363.0,387.382629,14.940377,"POLYGON ((34.73762 -0.00081, 34.73762 0, 34.73..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549215,27,19795,1896.60,27.21,19795,34.758152,-0.153262,34.757652,-0.153666,34.758651,-0.152857,0.0,0.0,21.023420,NaN,"POLYGON ((34.75865 -0.15367, 34.75865 -0.15286..."
549216,27,19796,1891.80,27.17,19796,34.759150,-0.153262,34.758650,-0.153666,34.759649,-0.152857,0.0,0.0,21.023420,NaN,"POLYGON ((34.75965 -0.15367, 34.75965 -0.15286..."
549217,27,19797,1881.36,27.08,19797,34.760147,-0.153262,34.759648,-0.153666,34.760647,-0.152857,0.0,0.0,NaN,NaN,"POLYGON ((34.76065 -0.15367, 34.76065 -0.15286..."
549218,27,19798,1870.92,27.00,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,0.0,0.0,NaN,NaN,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286..."


In [11]:
pop_centroids_hcf.columns

Index(['origin_id', 'destination_id', 'duration_seconds', 'distance_km',
       'grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max',
       'lat_max', 'bcount', 'pop_grid_bcount', 'pop_grid_pop', 'pop',
       'geometry'],
      dtype='object')

In [12]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    #"rowid": "grid_id",
    "origin_id": "hcf_uid",
    "pop": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "hcf_uid", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

In [13]:
pop_centroids_hcf

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_uid,duration_seconds,distance_km
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,NaN,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73...",0,698.76,12.26
1,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,NaN,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73...",0,685.44,12.15
2,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,1.067170,1.0,363.0,387.382629,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73...",0,674.14,12.05
3,3,34.736124,-0.000404,34.735625,-0.000809,34.736623,0.000000,5.335849,5.0,363.0,387.382629,"POLYGON ((34.73662 -0.00081, 34.73662 0, 34.73...",0,651.94,11.87
4,4,34.737122,-0.000404,34.736623,-0.000809,34.737621,0.000000,14.940377,14.0,363.0,387.382629,"POLYGON ((34.73762 -0.00081, 34.73762 0, 34.73...",0,639.37,11.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549215,19795,34.758152,-0.153262,34.757652,-0.153666,34.758651,-0.152857,NaN,0.0,0.0,21.023420,"POLYGON ((34.75865 -0.15367, 34.75865 -0.15286...",27,1896.60,27.21
549216,19796,34.759150,-0.153262,34.758650,-0.153666,34.759649,-0.152857,NaN,0.0,0.0,21.023420,"POLYGON ((34.75965 -0.15367, 34.75965 -0.15286...",27,1891.80,27.17
549217,19797,34.760147,-0.153262,34.759648,-0.153666,34.760647,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76065 -0.15367, 34.76065 -0.15286...",27,1881.36,27.08
549218,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286...",27,1870.92,27.00


Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [14]:
pop_centroids_hcf.columns

Index(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min',
       'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'population',
       'bcount', 'pop_grid_bcount', 'pop_grid_pop', 'geometry', 'hcf_uid',
       'duration_seconds', 'distance_km'],
      dtype='object')

In [17]:
healthcare_facilities_validated.columns

Index(['field_1', 'Field_1_1', 'Admin_1', 'facility_name', 'facility_type',
       'owner_type_name', 'latitude', 'longitude', 'Sub_Couinty', 'LL_Source',
       'ward_name', 'keph_level_name', 'Operating_Time',
       'total_ inpatient_beds', 'general_inpatient_beds', 'cots',
       'maternity_beds', 'emergency_casualty_beds', 'intensive_care_unit_beds',
       'high_dependancy_unit_beds', 'isolation_beds', 'general_theatres',
       'maternity_theatres', 'minor_theatres', 'BMoc', 'CeMoc',
       'Local_Validation', 'hcf_id', 'geometry'],
      dtype='object')

In [18]:
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities_validated[['hcf_id','facility_name', 'longitude', 'latitude', 'Local_Validation']], 
                     left_on='hcf_uid', right_on='hcf_id', how='left') # left on is from od matrix, right on is from healthcare facilities

In [19]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "longitude": "dest_lon",
    "latitude": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['hcf_uid'])

In [20]:
category_counts = healthcare_facilities_validated['Local_Validation'].value_counts()
print(category_counts)

Local_Validation
Private Comprehensive EmOC    11
Private Basic EmOC            10
Public Basic EmOC              5
Public Comprehensive EmOC      2
Name: count, dtype: int64


In [21]:
distances_duration_matrix['Local_Validation'].value_counts()

Local_Validation
Private Comprehensive EmOC    215765
Private Basic EmOC            196150
Public Basic EmOC              98075
Public Comprehensive EmOC      39230
Name: count, dtype: int64

In [22]:
selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC', 
                       'Private Basic EmOC', 'Public Basic EmOC']

In [23]:
distances_duration_matrix = distances_duration_matrix[
    distances_duration_matrix['Local_Validation'].isin(selected_categories)]

distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,NaN,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73...",698.76,12.26,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC
1,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,NaN,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73...",685.44,12.15,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC
2,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,1.067170,1.0,363.0,387.382629,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73...",674.14,12.05,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC
3,3,34.736124,-0.000404,34.735625,-0.000809,34.736623,0.000000,5.335849,5.0,363.0,387.382629,"POLYGON ((34.73662 -0.00081, 34.73662 0, 34.73...",651.94,11.87,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC
4,4,34.737122,-0.000404,34.736623,-0.000809,34.737621,0.000000,14.940377,14.0,363.0,387.382629,"POLYGON ((34.73762 -0.00081, 34.73762 0, 34.73...",639.37,11.76,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549215,19795,34.758152,-0.153262,34.757652,-0.153666,34.758651,-0.152857,NaN,0.0,0.0,21.023420,"POLYGON ((34.75865 -0.15367, 34.75865 -0.15286...",1896.60,27.21,27,Geta Health Centre,34.69332,-0.05006,Public Basic EmOC
549216,19796,34.759150,-0.153262,34.758650,-0.153666,34.759649,-0.152857,NaN,0.0,0.0,21.023420,"POLYGON ((34.75965 -0.15367, 34.75965 -0.15286...",1891.80,27.17,27,Geta Health Centre,34.69332,-0.05006,Public Basic EmOC
549217,19797,34.760147,-0.153262,34.759648,-0.153666,34.760647,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76065 -0.15367, 34.76065 -0.15286...",1881.36,27.08,27,Geta Health Centre,34.69332,-0.05006,Public Basic EmOC
549218,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286...",1870.92,27.00,27,Geta Health Centre,34.69332,-0.05006,Public Basic EmOC


In [24]:
# creat subsets based on categories of 'Validation of HCFs Categorization'
categories = {
    "public_comprehensive_EmOC": ["Public Comprehensive EmOC"],
    "private_comprehensive_EmOC": ["Private Comprehensive EmOC"],
    "private_basic_EmOC": ["Private Basic EmOC"],
    "public_basic_EmOC": ["Public Basic EmOC"]
}

subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['Local_Validation'].str.contains('|'.join(values), na=False)
    ]
    for key, values in categories.items()
}

public_CEmOC = subsets["public_comprehensive_EmOC"]
private_CEmOC = subsets["private_comprehensive_EmOC"]
public_BEmOC = subsets["public_basic_EmOC"]
private_BEmOC = subsets["private_basic_EmOC"]

In [25]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)

In [26]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)
public_BEmOC_closest_3 = get_closest_3(public_BEmOC)
private_BEmOC_closest_3 = get_closest_3(private_BEmOC)

/var/folders/_0/jy_1jlp91v34q4g91twj01m80000gp/T/ipykernel_58681/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)
/var/folders/_0/jy_1jlp91v34q4g91twj01m80000gp/T/ipykernel_58681/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsm

In [27]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    public_CEmOC_closest_3, private_CEmOC_closest_3,
    public_BEmOC_closest_3, private_BEmOC_closest_3
])
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,NaN,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73...",725.28,12.86,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC
1,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,NaN,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73...",823.74,14.82,15,Kisumu County Referral Hospital,34.75600,-0.10181,Public Comprehensive EmOC
2,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,NaN,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73...",711.96,12.75,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC
3,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,NaN,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73...",810.42,14.71,15,Kisumu County Referral Hospital,34.75600,-0.10181,Public Comprehensive EmOC
4,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,1.06717,1.0,363.0,387.382629,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73...",700.66,12.65,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58840,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286...",926.84,11.42,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC
58841,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286...",985.39,12.49,5,Ambercare Medical Care,34.76384,-0.08458,Private Basic EmOC
58842,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76264 -0.15367, 34.76264 -0.15286...",818.50,10.88,7,St. Francis Hillside Medicare,34.75251,-0.10008,Private Basic EmOC
58843,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,NaN,"POLYGON ((34.76264 -0.15367, 34.76264 -0.15286...",861.88,10.88,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC


In [28]:
geometry = [Point(xy) for xy in zip(distances_duration_matrix['origin_lon'], distances_duration_matrix['origin_lat'])]
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry=geometry, crs="EPSG:4326")

In [29]:
gpkg_path = data_temp + 'distances_duration_3_closet_Emoc.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_3_closet_Emoc", driver="GPKG")

In [30]:
# Review and remove
origin_dest = distances_duration_matrix

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [31]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [32]:
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0        0   34.733131   -0.000404       34.732632       -0.000809   
1        0   34.733131   -0.000404       34.732632       -0.000809   
2        1   34.734128   -0.000404       34.733629       -0.000809   
3        1   34.734128   -0.000404       34.733629       -0.000809   
4        2   34.735126   -0.000404       34.734627       -0.000809   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0       34.733629             0.0         NaN     7.0             32.0   
1       34.733629             0.0         NaN     7.0             32.0   
2       34.734627             0.0         NaN    10.0             32.0   
3       34.734627             0.0         NaN    10.0             32.0   
4       34.735625             0.0     1.06717     1.0            363.0   

   pop_grid_pop                                           geometry  \
0           NaN  POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73.

In [33]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [34]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [35]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [36]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [37]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,Weight,Pop_W
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,...,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73...",725.28,12.86,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC,0.001196,NaN
1,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,...,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73...",823.74,14.82,15,Kisumu County Referral Hospital,34.75600,-0.10181,Public Comprehensive EmOC,0.000170,NaN
2,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,...,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73...",711.96,12.75,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC,0.001527,NaN
3,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,...,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73...",810.42,14.71,15,Kisumu County Referral Hospital,34.75600,-0.10181,Public Comprehensive EmOC,0.000225,NaN
4,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,1.06717,1.0,363.0,...,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73...",700.66,12.65,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC,0.001873,0.001999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58840,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,...,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286...",926.84,11.42,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC,0.000017,NaN
58841,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,...,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286...",985.39,12.49,5,Ambercare Medical Care,34.76384,-0.08458,Private Basic EmOC,0.000004,NaN
58842,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,...,"POLYGON ((34.76264 -0.15367, 34.76264 -0.15286...",818.50,10.88,7,St. Francis Hillside Medicare,34.75251,-0.10008,Private Basic EmOC,0.000190,NaN
58843,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,...,"POLYGON ((34.76264 -0.15367, 34.76264 -0.15286...",861.88,10.88,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC,0.000075,NaN


In [38]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()

In [39]:
origin_dest_sum

,hcf_id,Pop_W
0,0,33352.606219
1,1,13467.195565
2,2,20961.718944
3,3,32349.301906
4,4,4419.519696
5,5,19808.356214
6,6,2457.717355
7,7,15096.400945
8,8,17242.693796
9,9,3468.182219


In [40]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')

In [41]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_y
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,...,725.28,12.86,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC,0.001196,NaN,32349.301906
1,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,...,823.74,14.82,15,Kisumu County Referral Hospital,34.75600,-0.10181,Public Comprehensive EmOC,0.000170,NaN,26847.921277
2,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,...,711.96,12.75,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC,0.001527,NaN,32349.301906
3,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,...,810.42,14.71,15,Kisumu County Referral Hospital,34.75600,-0.10181,Public Comprehensive EmOC,0.000225,NaN,26847.921277
4,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,1.06717,1.0,363.0,...,700.66,12.65,3,Jaramogi Odinga Teaching & Referral Hospital,34.77038,-0.08855,Public Comprehensive EmOC,0.001873,0.001999,32349.301906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215760,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,...,926.84,11.42,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC,0.000017,NaN,33352.606219
215761,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,...,985.39,12.49,5,Ambercare Medical Care,34.76384,-0.08458,Private Basic EmOC,0.000004,NaN,19808.356214
215762,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,...,818.50,10.88,7,St. Francis Hillside Medicare,34.75251,-0.10008,Private Basic EmOC,0.000190,NaN,15096.400945
215763,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,...,861.88,10.88,0,Macmohan Health Care Limited,34.77501,-0.08287,Private Basic EmOC,0.000075,NaN,33352.606219


In [42]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [43]:
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

In [44]:
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/_0/jy_1jlp91v34q4g91twj01m80000gp/T/ipykernel_58681/2370967217.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


In [45]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [46]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [47]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [48]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,...,-0.08855,Public Comprehensive EmOC,0.001196,NaN,32349.301906,1.00,0.000031,3.696061e-08,2.334148e-05,0.013395
1,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,NaN,7.0,32.0,...,-0.10181,Public Comprehensive EmOC,0.000170,NaN,26847.921277,1.00,0.000037,6.329727e-09,2.334148e-05,0.013395
2,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,...,-0.08855,Public Comprehensive EmOC,0.001527,NaN,32349.301906,1.00,0.000031,4.721678e-08,2.750962e-05,0.015787
3,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,NaN,10.0,32.0,...,-0.10181,Public Comprehensive EmOC,0.000225,NaN,26847.921277,1.00,0.000037,8.361914e-09,2.750962e-05,0.015787
4,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,1.06717,1.0,363.0,...,-0.08855,Public Comprehensive EmOC,0.001873,0.001999,32349.301906,1.00,0.000031,5.791315e-08,3.152942e-05,0.018095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215760,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,...,-0.08287,Private Basic EmOC,0.000017,NaN,33352.606219,0.35,0.000010,1.772425e-10,7.488685e-07,0.000428
215761,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,NaN,0.0,0.0,...,-0.08458,Private Basic EmOC,0.000004,NaN,19808.356214,0.35,0.000018,7.120732e-11,7.488685e-07,0.000428
215762,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,...,-0.10008,Private Basic EmOC,0.000190,NaN,15096.400945,0.35,0.000023,4.398300e-09,2.613516e-06,0.001498
215763,19799,34.762143,-0.153262,34.761644,-0.153666,34.762642,-0.152857,NaN,0.0,0.0,...,-0.08287,Private Basic EmOC,0.000075,NaN,33352.606219,0.35,0.000010,7.835819e-10,2.613516e-06,0.001498


In [49]:
max(origin_dest_acc.Accessibility_standard)

1.0

In [50]:
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [52]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

In [53]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [54]:
# Group by multiple columns and calculate the mean for numeric columns
# results_grid = results_grid.groupby(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard']).count().reset_index()
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])
type(results_grid)

geopandas.geodataframe.GeoDataFrame

In [55]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access', driver='GPKG')

In [56]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,0.013395,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73..."
2,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,0.015787,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73..."
4,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,0.018095,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73..."
6,3,34.736124,-0.000404,34.735625,-0.000809,34.736623,0.000000,0.023469,"POLYGON ((34.73662 -0.00081, 34.73662 0, 34.73..."
8,4,34.737122,-0.000404,34.736623,-0.000809,34.737621,0.000000,0.027074,"POLYGON ((34.73762 -0.00081, 34.73762 0, 34.73..."
...,...,...,...,...,...,...,...,...,...
39220,19795,34.758152,-0.153262,34.757652,-0.153666,34.758651,-0.152857,0.000253,"POLYGON ((34.75865 -0.15367, 34.75865 -0.15286..."
39222,19796,34.759150,-0.153262,34.758650,-0.153666,34.759649,-0.152857,0.000279,"POLYGON ((34.75965 -0.15367, 34.75965 -0.15286..."
39224,19797,34.760147,-0.153262,34.759648,-0.153666,34.760647,-0.152857,0.000346,"POLYGON ((34.76065 -0.15367, 34.76065 -0.15286..."
39226,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,0.000428,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286..."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [57]:
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] > 0.000001, 'result'] = 2
results_grid.loc[results_grid['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.02, 'result'] = 0

In [58]:
category_counts = results_grid['result'].value_counts()
print(category_counts)

result
 0    12645
 2     3512
 1     3456
-1        2
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [60]:
results_grid.columns

Index(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min',
       'origin_lat_min', 'origin_lon_max', 'origin_lat_max',
       'Accessibility_standard', 'geometry', 'result'],
      dtype='object')

In [61]:
results_grid['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid.loc[(results_grid['Accessibility_standard'] > 0.000001) & (results_grid['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.003) & (results_grid['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.019) & (results_grid['Accessibility_standard'] < 0.03), 'focused'] = 1

In [62]:
category_counts = results_grid['focused'].value_counts()
print(category_counts)

focused
0    16987
1     2628
Name: count, dtype: int64


In [63]:
results_grid = results_grid.loc[results_grid['result'] != -1]

In [64]:
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [65]:
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,result,focused
0,0,34.733131,-0.000404,34.732632,-0.000809,34.733629,0.000000,0.013395,"POLYGON ((34.73363 -0.00081, 34.73363 0, 34.73...",1,0
2,1,34.734128,-0.000404,34.733629,-0.000809,34.734627,0.000000,0.015787,"POLYGON ((34.73463 -0.00081, 34.73463 0, 34.73...",1,0
4,2,34.735126,-0.000404,34.734627,-0.000809,34.735625,0.000000,0.018095,"POLYGON ((34.73563 -0.00081, 34.73563 0, 34.73...",1,0
6,3,34.736124,-0.000404,34.735625,-0.000809,34.736623,0.000000,0.023469,"POLYGON ((34.73662 -0.00081, 34.73662 0, 34.73...",0,1
8,4,34.737122,-0.000404,34.736623,-0.000809,34.737621,0.000000,0.027074,"POLYGON ((34.73762 -0.00081, 34.73762 0, 34.73...",0,1
...,...,...,...,...,...,...,...,...,...,...,...
39220,19795,34.758152,-0.153262,34.757652,-0.153666,34.758651,-0.152857,0.000253,"POLYGON ((34.75865 -0.15367, 34.75865 -0.15286...",2,0
39222,19796,34.759150,-0.153262,34.758650,-0.153666,34.759649,-0.152857,0.000279,"POLYGON ((34.75965 -0.15367, 34.75965 -0.15286...",2,0
39224,19797,34.760147,-0.153262,34.759648,-0.153666,34.760647,-0.152857,0.000346,"POLYGON ((34.76065 -0.15367, 34.76065 -0.15286...",2,0
39226,19798,34.761145,-0.153262,34.760646,-0.153666,34.761644,-0.152857,0.000428,"POLYGON ((34.76164 -0.15367, 34.76164 -0.15286...",2,0


In [66]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access-class', driver='GPKG')

In [67]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)